# Quantum Error Correction with Qiskit: from three qubits to the Steane code

**Qiskit Fall Fest workshop notebook.** Written for Qiskit 2.x, runs in Google Colab, no account needed.

This notebook follows *Quantum Computing Essentials with Qiskit*. It assumes you know gates and the Bloch sphere (Essentials, Chapters 2 and 3), measurement (Chapter 4), entanglement (Chapter 6) and noise (Chapter 8). Chapter 8 of the Essentials notebook ended with a sobering plot: the success rate of a GHZ circuit falling as the circuit grew. This notebook is about the only known way to stop that from happening at scale.

---

## How to read this notebook

Same rhythm as before. Each chapter follows three steps:

1. We **look** at something in code (a circuit, a set of states, a histogram, a plot).
2. We **explain** the mathematics behind what we just saw.
3. We **stop and ask you to try** something yourself.

The same conventions are used throughout:

> **Predict.** Before running a cell, write down what you expect to see. Being wrong is the useful part.

> **Physics note.** A short historical or physical aside that connects the code to the real laboratory.

> **Check.** A cell that tests your own answer and tells you whether it is right.

Exercises have hints and full solutions hidden inside collapsible blocks. Open them only after a real attempt.

## The story in one paragraph

Classical computers fight noise by keeping copies and taking a majority vote. Quantum mechanics seems to forbid all three ingredients: you cannot copy an unknown state, you cannot look at it without disturbing it, and its errors are continuous rather than simple flips. We get around all three. We build a **logical qubit** out of three physical qubits that survives any single bit flip ($X$ error), then a second one that survives any single phase flip ($Z$ error), and we discover along the way that correcting these two discrete errors is enough to correct *every* error. Then we leave you with a challenge: protect against both at once. We finish with the **Steane code**, a seven-qubit code built from a classical Hamming code, and build its complete error-correction cycle in Qiskit.

## Contents

| Chapter | Topic |
|---|---|
| 0 | Setting up |
| 1 | Why quantum error correction looks impossible |
| 2 | The bit-flip code: encoding a logical qubit |
| 3 | Syndrome measurement: looking without looking |
| 4 | The full correction cycle |
| 5 | The phase-flip code |
| 6 | Continuous errors become discrete |
| 7 | The challenge: both errors at once |
| 8 | The stabiliser language |
| 9 | The Steane code: theory |
| 10 | The Steane code in Qiskit |
| | Where we go next, and further reading |

## Chapter 0. Setting up

The same three packages as in the Essentials notebook. In this notebook we lean on `qiskit-aer` more than before, because error correction needs **mid-circuit measurements** and **classically controlled gates** (measure a qubit, then decide which gate to apply), and Aer simulates those exactly.

(Working locally instead of Colab? With `uv` you can skip the install cell and start Jupyter in one line: `uv run --with "qiskit[visualization]" --with qiskit-aer --with qiskit-ibm-runtime --with pylatexenc --with ipywidgets --with jupyterlab jupyter lab`.)

In [ ]:
%pip install -q "qiskit[visualization]" qiskit-aer qiskit-ibm-runtime pylatexenc ipywidgets

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt

import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, Operator, Pauli, SparsePauliOp
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_aer.noise import NoiseModel, pauli_error
from ipywidgets import interact, FloatSlider

print("Qiskit version:", qiskit.__version__)

aer_sampler = AerSampler()        # supports mid-circuit measurement and if_test


# The two helpers from the Essentials notebook.

def fig_to_img(fig):
    # Render a matplotlib Figure to an image array so we can place
    # several Bloch-sphere figures side by side.
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    buf.seek(0)
    return plt.imread(buf)


def check(label, test):
    # Run a test (a function returning True/False) and report the result.
    # If your variable is still None or the code raises, we say "Not yet".
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("Correct:  " if ok else "Not yet:  ") + label)


def noisy_sampler(p_x=0.0, p_y=0.0, p_z=0.0, method="automatic"):
    # A sampler whose 'id' gates apply X, Y or Z with the given probabilities.
    # We place 'id' gates exactly where we want errors to happen, and nowhere else.
    nm = NoiseModel()
    err = pauli_error([("X", p_x), ("Y", p_y), ("Z", p_z), ("I", 1 - p_x - p_y - p_z)])
    nm.add_all_qubit_quantum_error(err, ["id"])
    return AerSampler(options=dict(backend_options=dict(noise_model=nm, method=method)))

## Chapter 1. Why quantum error correction looks impossible

### The classical solution

To send one bit over a noisy channel that flips each bit with probability $p$, send it three times and take the majority:

$$0 \mapsto 000, \qquad 1 \mapsto 111 .$$

The vote is wrong only if two or three of the copies flip, so the **logical** error probability is

$$p_L = 3p^2(1-p) + p^3 = 3p^2 - 2p^3 .$$

For small $p$ this is roughly $3p^2$, much smaller than $p$. Below $p = 1/2$ the code always helps.

In [ ]:
rng = np.random.default_rng(0)

def classical_repetition(bit, p, shots):
    encoded = np.full((shots, 3), bit)
    flips = rng.random((shots, 3)) < p                  # each copy flips independently
    received = encoded ^ flips
    decoded = (received.sum(axis=1) >= 2).astype(int)   # majority vote
    return np.mean(decoded != bit)

ps = np.linspace(0, 0.5, 21)
sim = [classical_repetition(0, p, 20000) for p in ps]

plt.figure(figsize=(7, 3.8))
plt.plot(ps, ps, "--", c="gray", label="no encoding: $p$")
plt.plot(ps, 3 * ps**2 - 2 * ps**3, label="theory: $3p^2 - 2p^3$")
plt.plot(ps, sim, "o", label="simulated majority vote")
plt.xlabel("physical error probability p"); plt.ylabel("logical error probability")
plt.title("The classical repetition code"); plt.legend(); plt.grid(alpha=.3); plt.show()

### Three obstacles

Try to copy this idea to qubits and you hit three walls at once.

1. **No cloning.** There is no circuit that maps $|\psi\rangle|0\rangle|0\rangle \to |\psi\rangle|\psi\rangle|\psi\rangle$ for an unknown $|\psi\rangle$. (If there were, it would be linear, and a linear map cannot square amplitudes.) So "send three copies" is out.
2. **Measurement disturbs.** The majority vote *reads* the bits. Reading a qubit in superposition collapses it, destroying the very amplitudes $\alpha$ and $\beta$ we want to protect.
3. **Errors are continuous.** A classical bit either flips or it does not. A qubit can be rotated by a tiny angle, $R_x(0.01)$, or pick up a small relative phase. There are infinitely many possible errors. How could a finite code fix them all?

> **Physics note.** In the early 1990s many physicists, including Rolf Landauer and Bill Unruh, argued that decoherence would make large quantum computers impossible for exactly these reasons. The no-cloning theorem had been proved in 1982 by Wootters and Zurek, and independently by Dieks. Then in 1995 Peter Shor, a year after his factoring algorithm, published a nine-qubit code that defeats all three obstacles. Andrew Steane found the seven-qubit code independently in 1996. This notebook rebuilds their ideas in the order they are easiest to understand.

### Exercise 1: the classical numbers

**Goal.** For $p = 0.1$, compute the logical error probability of the three-bit repetition code and store it as `ex1_pL`.

<details>
<summary>Hint</summary>

$3p^2 - 2p^3$. Compare the answer with $p$ itself: how much has encoding helped?
</details>

In [ ]:
ex1_pL = None

In [ ]:
check("logical error probability at p = 0.1", lambda: np.isclose(ex1_pL, 0.028))

<details>
<summary>Solution</summary>

```python
ex1_pL = 3 * 0.1**2 - 2 * 0.1**3      # 0.028
```

An error rate of 10% becomes 2.8%, more than three times better. At $p = 0.01$ the improvement is more than thirty times ($p_L \approx 0.0003$). Encoding helps most when the physical error rate is already small. Keep this in mind: it is the seed of the **threshold theorem** at the end of the notebook.
</details>

Further reading: the IBM Quantum Learning course [Foundations of quantum error correction](https://quantum.cloud.ibm.com/learning/en/courses/foundations-of-quantum-error-correction) develops all of this notebook in more depth.

## Chapter 2. The bit-flip code: encoding a logical qubit

We do not copy the *state*. We copy the *basis*. Define a **logical qubit** by

$$|0_L\rangle = |000\rangle, \qquad |1_L\rangle = |111\rangle,$$

and encode an arbitrary state by linearity:

$$\alpha|0\rangle + \beta|1\rangle \ \longmapsto\ \alpha|000\rangle + \beta|111\rangle .$$

This is **not** three copies. Three copies would be $(\alpha|0\rangle+\beta|1\rangle)^{\otimes 3}$, which has eight terms. Our encoded state has two, and it is entangled. Each physical qubit on its own knows nothing about $\alpha$ and $\beta$; the information lives in the correlations, exactly like the Bell and GHZ states of Essentials Chapter 6.

The encoder is two CNOTs. The state to protect starts on qubit 0, and qubits 1 and 2 start in $|0\rangle$:

$$(\alpha|0\rangle+\beta|1\rangle)|00\rangle \xrightarrow{\ \text{CNOT}_{0\to1}\ } \xrightarrow{\ \text{CNOT}_{0\to2}\ } \alpha|000\rangle + \beta|111\rangle .$$

Each CNOT copies the *basis value* of qubit 0, which is allowed, rather than the state, which is not.

In [ ]:
def bitflip_encoder():
    qc = QuantumCircuit(3, name="enc_X")
    qc.cx(0, 1)
    qc.cx(0, 2)
    return qc

theta = 2 * np.arccos(np.sqrt(0.7))          # alpha = sqrt(0.7), beta = sqrt(0.3)
qc = QuantumCircuit(3)
qc.ry(theta, 0)
qc.barrier()
qc.compose(bitflip_encoder(), inplace=True)
display(qc.draw("mpl"))

sv = Statevector(qc)
display(sv.draw("latex"))
print("probabilities:", {str(k): round(float(v), 3) for k, v in sv.probabilities_dict().items()})

Two terms, `000` and `111`, with probabilities 0.7 and 0.3: the original $|\alpha|^2$ and $|\beta|^2$, now spread across three qubits.

> **Predict.** Look at the Bloch spheres of the three physical qubits. Will any of them still show the original arrow?

In [ ]:
plot_bloch_multivector(sv, title="encoded state: each physical qubit on its own")

All three arrows have shrunk onto the vertical axis, to $z = 0.7 - 0.3 = 0.4$. Each qubit's $Z$ statistics survive (every qubit is 0 with probability 0.7) but the phase information, the $x$ and $y$ components, is gone from every individual qubit. It has not been lost; it lives only in the joint state.

### Exercise 2: encoding $|-\rangle$

**Goal.** Build a three-qubit circuit `ex2_circuit` that prepares $|-\rangle$ on qubit 0 and encodes it. Then answer in a comment: is the result $|-\rangle|-\rangle|-\rangle$?

<details>
<summary>Hint</summary>

$|-\rangle = HX|0\rangle$: apply `x` then `h`. By linearity the encoded state is $\frac{1}{\sqrt2}(|000\rangle - |111\rangle)$. Expand $|{-}{-}{-}\rangle$ and count its terms.
</details>

In [ ]:
ex2_circuit = None

In [ ]:
check("ex2_circuit prepares (|000> - |111>)/sqrt2",
      lambda: Statevector(ex2_circuit).equiv(Statevector(np.array([1, 0, 0, 0, 0, 0, 0, -1]) / np.sqrt(2))))

<details>
<summary>Solution</summary>

```python
ex2_circuit = QuantumCircuit(3)
ex2_circuit.x(0)
ex2_circuit.h(0)
ex2_circuit.compose(bitflip_encoder(), inplace=True)
# Not |--->. That state has eight terms; ours has two and is a GHZ state with a minus sign.
```

The encoded $|\pm_L\rangle$ are the GHZ states $\frac{1}{\sqrt2}(|000\rangle \pm |111\rangle)$. Remember this: in Chapter 5 we will see that a phase flip on *any one* of the three qubits turns $|+_L\rangle$ into $|-_L\rangle$. That is the weakness of this code.
</details>

## Chapter 3. Syndrome measurement: looking without looking

Suppose one qubit suffers an $X$ error. The state becomes one of

| error | state |
|---|---|
| none | $\alpha|000\rangle + \beta|111\rangle$ |
| $X_0$ | $\alpha|001\rangle + \beta|110\rangle$ |
| $X_1$ | $\alpha|010\rangle + \beta|101\rangle$ |
| $X_2$ | $\alpha|100\rangle + \beta|011\rangle$ |

(Qiskit order: qubit 0 is the rightmost bit.) A majority vote would measure all three qubits and destroy $\alpha$ and $\beta$. But notice what distinguishes the rows: not the value of any qubit, but **which qubits agree with each other**. In both branches of each row, the pattern of agreements is the same.

So we measure only **parities**:

- Do qubits 0 and 1 agree? That is the eigenvalue of $Z_0Z_1$: $+1$ if they agree, $-1$ if not.
- Do qubits 1 and 2 agree? That is the eigenvalue of $Z_1Z_2$.

Both branches $|000\rangle$ and $|111\rangle$ give the same answers, so measuring these parities tells us **where the error is** and **nothing about the encoded state**. The two answers together are called the **syndrome**.

| syndrome $(Z_0Z_1, Z_1Z_2)$ | as bits (anc1 anc0) | diagnosis |
|---|---|---|
| $(+1, +1)$ | `00` | no error |
| $(-1, +1)$ | `01` | $X$ on qubit 0 |
| $(-1, -1)$ | `11` | $X$ on qubit 1 |
| $(+1, -1)$ | `10` | $X$ on qubit 2 |

### How to measure a parity

Use a fresh **ancilla** qubit in $|0\rangle$ and CNOT both data qubits into it. The ancilla ends in $|x_0\oplus x_1\rangle$, the parity, in *both* branches of the superposition. Measuring the ancilla reads the parity without touching the data qubits' superposition.

> **Physics note.** Comparing bits instead of reading them is the oldest idea in coding theory. Richard Hamming invented parity-check codes at Bell Labs in 1950, frustrated that the relay computer kept halting on weekends because of single-bit errors in its punched-card input. The quantum version adds one essential twist: the parity must be measured *collectively*. Measuring qubit 0 and qubit 1 separately and then comparing would also give the parity, but it would collapse the state. The ancilla extracts the comparison and only the comparison.

In [ ]:
def bitflip_syndrome_circuit(error_qubit=None, theta=0.0):
    data = QuantumRegister(3, "data")
    anc  = QuantumRegister(2, "anc")
    syn  = ClassicalRegister(2, "syn")
    qc = QuantumCircuit(data, anc, syn)

    qc.ry(theta, data[0])                              # the state to protect
    qc.compose(bitflip_encoder(), data, inplace=True)
    qc.barrier()
    if error_qubit is not None:
        qc.x(data[error_qubit])                        # inject one X error
    qc.barrier()
    qc.cx(data[0], anc[0]); qc.cx(data[1], anc[0])     # anc0 = parity of qubits 0, 1  (Z0 Z1)
    qc.cx(data[1], anc[1]); qc.cx(data[2], anc[1])     # anc1 = parity of qubits 1, 2  (Z1 Z2)
    qc.measure(anc, syn)
    return qc

bitflip_syndrome_circuit(error_qubit=1, theta=1.0).draw("mpl")

> **Predict.** Using the table above, which syndrome should each of the four cases give?

In [ ]:
for e in [None, 0, 1, 2]:
    counts = aer_sampler.run([bitflip_syndrome_circuit(e, theta=1.0)], shots=1000).result()[0].data.syn.get_counts()
    print(f"error on qubit {e}:  syndrome counts {counts}")

Every case gives one syndrome with certainty. Now the crucial check: the syndrome must not depend on the encoded state. We repeat the experiment for many different $\alpha, \beta$.

In [ ]:
for theta in [0.0, 0.7, np.pi / 2, 2.5, np.pi]:
    counts = aer_sampler.run([bitflip_syndrome_circuit(1, theta)], shots=500).result()[0].data.syn.get_counts()
    print(f"theta = {theta:.2f}  (P(1) = {np.sin(theta/2)**2:.2f}):  syndrome counts {counts}")

Same syndrome every time. The measurement has learned where the error is and nothing at all about the state.

### Exercise 3: the decoder as a lookup table

**Goal.** Write the decoder as a Python dictionary `ex3_lookup` that maps each syndrome string (as Qiskit prints it, `"anc1 anc0"`) to the index of the qubit to correct, or to `None` for no correction.

<details>
<summary>Hint</summary>

Read it off the table above. Bit `anc0` is the rightmost character.
</details>

In [ ]:
ex3_lookup = None      # {"00": ..., "01": ..., "10": ..., "11": ...}

In [ ]:
def _ex3():
    for e in [None, 0, 1, 2]:
        s = list(aer_sampler.run([bitflip_syndrome_circuit(e, 0.4)], shots=10).result()[0].data.syn.get_counts())[0]
        if ex3_lookup[s] != e:
            return False
    return True

check("your lookup table identifies every single X error", _ex3)

<details>
<summary>Solution</summary>

```python
ex3_lookup = {"00": None, "01": 0, "11": 1, "10": 2}
```

A lookup table from syndromes to corrections is called a **decoder**. For three qubits it has four entries. For large codes the table is astronomically big, and building fast decoders is a research field of its own.
</details>

## Chapter 4. The full correction cycle

Now we close the loop: encode, suffer an error, measure the syndrome, **apply the correction**, and check that the original state is back.

The correction depends on a measurement outcome, so we use a **classically controlled gate**. In Qiskit 2.x this is written with `if_test`:

```python
with qc.if_test((syn, 1)):     # if the syndrome register reads the integer 1 ("01")
    qc.x(data[0])
```

To check the result we **decode** (run the encoder backwards) and undo the preparation rotation. If the error was corrected, all three data qubits are back in $|0\rangle$ and we measure `000` every time.

The function below builds the whole cycle for both codes in this notebook. The errors are given as a list like `[("x", 1)]` or `[("rx", 1, 0.3)]`. It also places an `id` gate on every data qubit, which does nothing on an ideal simulator but is where our noise model will put random errors later.

In [ ]:
def repetition_cycle(code="bit", theta=0.0, errors=(), correct=True):
    # code = "bit" (protects against X) or "phase" (protects against Z, Chapter 5)
    data = QuantumRegister(3, "data")
    anc  = QuantumRegister(2, "anc")
    syn  = ClassicalRegister(2, "syn")
    out  = ClassicalRegister(3, "out")
    qc = QuantumCircuit(data, anc, syn, out)

    enc = bitflip_encoder() if code == "bit" else phaseflip_encoder()

    # 1. prepare and encode
    qc.ry(theta, data[0])
    qc.compose(enc, data, inplace=True)
    qc.barrier()

    # 2. errors (deliberate ones, plus the 'id' slots for random noise)
    for name, q, *params in errors:
        getattr(qc, name)(*params, data[q])
    qc.id(data)
    qc.barrier()

    # 3. syndrome measurement
    for a, (i, j) in enumerate([(0, 1), (1, 2)]):
        if code == "bit":                               # Z_i Z_j parity check
            qc.cx(data[i], anc[a]); qc.cx(data[j], anc[a])
        else:                                           # X_i X_j parity check (Chapter 5)
            qc.h(anc[a])
            qc.cx(anc[a], data[i]); qc.cx(anc[a], data[j])
            qc.h(anc[a])
    qc.measure(anc, syn)

    # 4. correction
    if correct:
        fix = qc.x if code == "bit" else qc.z
        for s, q in [(1, 0), (3, 1), (2, 2)]:           # the lookup table from Exercise 3
            with qc.if_test((syn, s)):
                fix(data[q])
    qc.barrier()

    # 5. decode and check
    qc.compose(enc.inverse(), data, inplace=True)
    qc.ry(-theta, data[0])
    qc.measure(data, out)                               # recovered means all three read 0
    return qc


def phaseflip_encoder():          # used from Chapter 5 on; defined here so the function above works
    qc = QuantumCircuit(3, name="enc_Z")
    qc.cx(0, 1); qc.cx(0, 2)
    qc.h([0, 1, 2])
    return qc


def run_cycle(qc, shots=1000, sampler=aer_sampler):
    res = sampler.run([qc], shots=shots).result()[0].data
    return res.syn.get_counts(), res.out.get_counts().get("000", 0) / shots


repetition_cycle("bit", theta=1.0, errors=[("x", 1)]).draw("mpl", fold=-1)

> **Predict.** With correction switched off, an $X$ error on one qubit should spoil the state. With it on, the state should come back perfectly. What probability of reading `out = 000` do you expect in each case, for $\theta = 1$?

In [ ]:
theta = 1.0
print(" error      | correction | syndrome      | P(state recovered)")
print("------------+------------+---------------+-------------------")
for err in [[], [("x", 0)], [("x", 1)], [("x", 2)]]:
    for corr in [False, True]:
        syn, p_ok = run_cycle(repetition_cycle("bit", theta, err, correct=corr))
        label = "none" if not err else f"X on q{err[0][1]}"
        print(f" {label:10s} | {str(corr):10s} | {str(syn):13s} | {p_ok:.3f}")

Without correction, an error is never undone: the decoded qubits do not return to `000`. With correction the state comes back with probability exactly 1 in every case.

### The logical error rate

Now let random noise act. Each data qubit independently suffers an $X$ error with probability $p$. The code fails only if two or more qubits flip, so we expect the **same curve as the classical repetition code**, $3p^2 - 2p^3$. We start from $|0_L\rangle$, so a logical failure always shows up as a wrong `out`.

In [ ]:
ps = np.array([0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5])
pL = []
for p in ps:
    _, p_ok = run_cycle(repetition_cycle("bit", 0.0), shots=4000, sampler=noisy_sampler(p_x=p))
    pL.append(1 - p_ok)

fine = np.linspace(0, 0.5, 100)
plt.figure(figsize=(7, 3.8))
plt.plot(fine, fine, "--", c="gray", label="unprotected qubit: $p$")
plt.plot(fine, 3 * fine**2 - 2 * fine**3, label="theory: $3p^2 - 2p^3$")
plt.plot(ps, pL, "o", label="bit-flip code, simulated")
plt.xlabel("X error probability per qubit"); plt.ylabel("logical error probability")
plt.title("The bit-flip code against random X errors"); plt.legend(); plt.grid(alpha=.3); plt.show()

> **Physics note.** On IBM hardware the `if_test` above is a real **dynamic circuit**: the qubits are measured mid-circuit, the result travels to classical control electronics, and the conditional gate is applied within about a microsecond, before the qubits decohere. In large-scale error correction one often does not apply the correction at all. Since the corrections are Pauli gates, the computer simply **records** them in software and reinterprets later measurement results. This is called **Pauli frame tracking**, and it removes the feed-forward delay from the critical path.

### Exercise 4: two errors

**Goal.** Predict what happens when qubits 0 and 1 *both* suffer $X$ errors. Store your predicted value of `P(state recovered)` for $\theta = 0$ in `ex4_p_recovered`, then confirm with `run_cycle`.

<details>
<summary>Hint</summary>

Work out the syndrome of $X_0X_1$ from the parities, then look up which correction the decoder applies. What is the total error after correction?
</details>

In [ ]:
ex4_p_recovered = None

In [ ]:
check("prediction for two X errors matches the simulation",
      lambda: np.isclose(ex4_p_recovered, run_cycle(repetition_cycle("bit", 0.0, [("x", 0), ("x", 1)]))[1]))

<details>
<summary>Solution</summary>

```python
ex4_p_recovered = 0.0
```

$X_0X_1$ makes qubits 0 and 1 agree with each other but disagree with qubit 2: syndrome `10`, which the decoder reads as "$X$ on qubit 2". After correction the total error is $X_0X_1X_2$, which maps $|000\rangle \leftrightarrow |111\rangle$. That is a **logical $X$**: the code has not just failed to help, it has turned the error into a perfectly valid-looking wrong state. The decoder always assumes the *most likely* error, and two errors are less likely than one, but not impossible.
</details>

Further reading: the IBM guide on [classical feedforward and control flow](https://quantum.cloud.ibm.com/docs/en/guides/classical-feedforward-and-control-flow), which documents `if_test` and how it runs on hardware.

## Chapter 5. The phase-flip code

### The weakness of the bit-flip code

A $Z$ error leaves $|0\rangle$ alone and flips the sign of $|1\rangle$. On the encoded state,

$$Z_0\big(\alpha|000\rangle + \beta|111\rangle\big) = \alpha|000\rangle - \beta|111\rangle .$$

That is exactly a logical $Z$. The same happens for $Z_1$ or $Z_2$. Worse, the parities $Z_0Z_1$ and $Z_1Z_2$ commute with $Z$ errors, so **the syndrome does not notice**.

> **Predict.** Run the bit-flip cycle on $|+\rangle$ (that is, $\theta = \pi/2$) with a $Z$ error on qubit 1. What syndrome do you expect, and what recovery probability?

In [ ]:
syn, p_ok = run_cycle(repetition_cycle("bit", np.pi / 2, [("z", 1)]))
print("syndrome:", syn, "   P(state recovered):", p_ok)

Silent failure: syndrome `00`, and the state has turned from $|+\rangle$ into $|-\rangle$, so it is never recovered. Phase errors are not exotic, either. They come from the **dephasing** ($T_2$) process of Essentials Chapter 8, which on most devices is at least as fast as energy relaxation.

### The fix: change basis

Essentials Chapter 3 showed $HZH = X$: in the $|\pm\rangle$ basis a phase flip looks exactly like a bit flip. So we run the bit-flip code in that basis.

$$|0_L\rangle = |{+}{+}{+}\rangle, \qquad |1_L\rangle = |{-}{-}{-}\rangle .$$

The encoder is the bit-flip encoder followed by a Hadamard on every qubit. A $Z$ error on any one qubit now flips $|+\rangle \leftrightarrow |-\rangle$ on that qubit, and we detect it with the parities

$$X_0X_1 \quad\text{and}\quad X_1X_2 ,$$

the bit-flip checks with every $Z$ replaced by $X$.

### Measuring an $X$-parity

To measure $X_iX_j$, prepare the ancilla in $|+\rangle$, apply CNOTs **from the ancilla to the data qubits**, then a Hadamard on the ancilla and measure. This is phase kickback (Deutsch-Jozsa notebook, Chapter 3): the CNOT applies $X_iX_j$ to the data conditioned on the ancilla, and the eigenvalue $\pm1$ of the data state is kicked back onto the ancilla's phase, where the final Hadamard turns it into a 0 or 1. It is also a one-bit phase estimation, if you have met that notebook.

`repetition_cycle("phase", ...)` from Chapter 4 already contains this. Here is the circuit.

In [ ]:
repetition_cycle("phase", theta=1.0, errors=[("z", 1)]).draw("mpl", fold=-1)

In [ ]:
theta = 1.0
print(" error      | syndrome      | P(state recovered)")
print("------------+---------------+-------------------")
for err in [[], [("z", 0)], [("z", 1)], [("z", 2)], [("x", 1)]]:
    syn, p_ok = run_cycle(repetition_cycle("phase", theta, err))
    label = "none" if not err else f"{err[0][0].upper()} on q{err[0][1]}"
    print(f" {label:10s} | {str(syn):13s} | {p_ok:.3f}")

Every single $Z$ error is corrected. But look at the last row: the phase-flip code is helpless against an $X$ error, for exactly the mirror-image reason. **Each code protects against one kind of error and is blind to the other.**

> **Physics note.** Some hardware is deliberately built with very lopsided noise. "Cat qubits", developed by Alice & Bob and AWS, store a qubit in two coherent states of a microwave resonator, and their bit-flip rate is suppressed exponentially as the resonator is driven harder, leaving mostly phase flips. On such **biased-noise** hardware, a phase-flip repetition code like this one does most of the work.

### Exercise 5: the phase-flip encoder by hand

**Goal.** Build your own three-qubit circuit `ex5_encoder` (input on qubit 0) such that $|0\rangle \mapsto |{+}{+}{+}\rangle$ and $|1\rangle \mapsto |{-}{-}{-}\rangle$. Do not call `phaseflip_encoder`.

<details>
<summary>Hint</summary>

First spread the basis value with two CNOTs, as for the bit-flip code. Then which gate turns $|0\rangle$ into $|+\rangle$ and $|1\rangle$ into $|-\rangle$ on every qubit?
</details>

In [ ]:
ex5_encoder = None

In [ ]:
def _ex5():
    zero = QuantumCircuit(3).compose(ex5_encoder)
    one = QuantumCircuit(3); one.x(0); one.compose(ex5_encoder, inplace=True)
    return (Statevector(zero).equiv(Statevector.from_label("+++"))
            and Statevector(one).equiv(Statevector.from_label("---")))

check("|0> -> |+++> and |1> -> |--->", _ex5)

<details>
<summary>Solution</summary>

```python
ex5_encoder = QuantumCircuit(3)
ex5_encoder.cx(0, 1)
ex5_encoder.cx(0, 2)
ex5_encoder.h([0, 1, 2])
```

The order matters: CNOT first, then Hadamards. Hadamards first would give a different (and useless) state.
</details>

## Chapter 6. Continuous errors become discrete

We have dealt with obstacles 1 (no cloning) and 2 (measurement). Obstacle 3 remains: real errors are continuous. Suppose qubit 1 of the bit-flip code is rotated by a small angle,

$$R_x(\epsilon) = \cos\tfrac{\epsilon}{2}\,I - i\sin\tfrac{\epsilon}{2}\,X .$$

The encoded state becomes a **superposition** of "no error" and "$X$ error on qubit 1":

$$\cos\tfrac{\epsilon}{2}\,|\psi_L\rangle \;-\; i\sin\tfrac{\epsilon}{2}\;X_1|\psi_L\rangle .$$

These two branches have *different syndromes*: `00` and `11`. Measuring the syndrome therefore **collapses** the superposition. With probability $\cos^2(\epsilon/2)$ we get `00` and the state is exactly $|\psi_L\rangle$, no error at all. With probability $\sin^2(\epsilon/2)$ we get `11` and the state is exactly $X_1|\psi_L\rangle$, which we know how to correct. Either way, the outcome is perfect.

**The syndrome measurement turns a continuous error into a discrete one.** The same argument works for any error: every $2\times2$ matrix can be written as

$$E = a\,I + b\,X + c\,Y + d\,Z ,$$

so any single-qubit error is a superposition of "nothing", $X$, $Y$ and $Z$. A code that can identify and correct those four discrete cases on a qubit can correct **every** error on that qubit, including all the continuous ones.

> **Physics note.** This is the conceptual heart of quantum error correction, and it is the reason it works at all. In the classical analogue-computing era, small continuous drifts accumulated without limit. Here, the measurement that seemed to be an enemy in Chapter 1 becomes the tool that digitises the noise. The same projective measurement is used in the lab every cycle on today's error-correction experiments.

We see it happen. We apply $R_x(\epsilon)$ to qubit 1 and record both the syndrome statistics and the recovery probability.

In [ ]:
eps_list = np.linspace(0, np.pi, 13)
p_fire, p_rec = [], []
for eps in eps_list:
    syn, p_ok = run_cycle(repetition_cycle("bit", 1.0, [("rx", 1, eps)]), shots=4000)
    p_fire.append(syn.get("11", 0) / 4000)
    p_rec.append(p_ok)

fine = np.linspace(0, np.pi, 200)
plt.figure(figsize=(7, 3.8))
plt.plot(eps_list, p_fire, "o", label="P(syndrome fires), simulated")
plt.plot(fine, np.sin(fine / 2)**2, label=r"theory $\sin^2(\epsilon/2)$")
plt.plot(eps_list, p_rec, "s", label="P(state recovered) after correction")
plt.xlabel(r"rotation angle $\epsilon$ of the error $R_x(\epsilon)$ on qubit 1"); plt.ylabel("probability")
plt.ylim(-0.05, 1.05); plt.legend(); plt.grid(alpha=.3)
plt.title("A continuous error, corrected perfectly"); plt.show()

The syndrome fires with probability $\sin^2(\epsilon/2)$, and the state is recovered **every time**, for every angle.

### Interactive: watch the collapse

Choose an error angle. The cell prints the encoded state right after the error, which is a superposition of the two branches, and the two possible states after the syndrome measurement, each of which is a clean codeword (or a clean single error).

In [ ]:
@interact(eps=FloatSlider(min=0, max=np.pi, step=0.05, value=0.6, description="epsilon"))
def collapse_demo(eps):
    qc = QuantumCircuit(3)
    qc.ry(1.0, 0)
    qc.compose(bitflip_encoder(), inplace=True)
    qc.rx(eps, 1)
    sv = Statevector(qc)
    print("after the error:")
    for k, a in sv.to_dict().items():
        if abs(a) > 1e-9:
            print(f"   {a:+.3f} |{k}>")
    w_good = sum(abs(sv.data[i])**2 for i in [0b000, 0b111])
    print(f"\nsyndrome 00 with probability {w_good:.3f}: state collapses to the uncorrupted codeword")
    print(f"syndrome 11 with probability {1 - w_good:.3f}: state collapses to X on qubit 1, which we then correct")

### Exercise 6: how often does the syndrome fire?

**Goal.** For an error $R_x(\pi/3)$ on one qubit of the bit-flip code, what is the probability that the syndrome reports an error? Store it as `ex6_p`.

<details>
<summary>Hint</summary>

$\sin^2(\epsilon/2)$ with $\epsilon = \pi/3$.
</details>

In [ ]:
ex6_p = None

In [ ]:
check("P(syndrome fires) for Rx(pi/3)", lambda: np.isclose(ex6_p, 0.25))

<details>
<summary>Solution</summary>

```python
ex6_p = np.sin(np.pi / 6)**2      # 0.25
```

Three times in four the measurement finds *no* error and the state is restored by the measurement alone; once in four it finds a full $X$ error and the correction restores it.
</details>

## Chapter 7. The challenge: both errors at once

Here is where we stand.

| | corrects one $X$ | corrects one $Z$ |
|---|---|---|
| bit-flip code, $\{|000\rangle, |111\rangle\}$ | yes | **no** |
| phase-flip code, $\{|{+}{+}{+}\rangle, |{-}{-}{-}\rangle\}$ | **no** | yes |

And from Chapter 6 we know that if a single code could correct $X$, $Z$ **and** $Y$ on any one qubit, it would correct *every* single-qubit error.

**The question.** How can we build one logical qubit that is protected against an arbitrary error on any one of its physical qubits?

Stop and think before opening the hints. Open them one at a time.

<details>
<summary>Hint 1: do you need to worry about Y separately?</summary>

$Y = iXZ$. A $Y$ error is an $X$ error and a $Z$ error on the *same* qubit (up to a phase, which is global and harmless). So if your code can detect and correct an $X$ and a $Z$ independently, even when they hit the same qubit, it handles $Y$ for free.
</details>

<details>
<summary>Hint 2: what if the "physical qubits" of one code were themselves protected?</summary>

The phase-flip code protects against $Z$ errors on its three qubits but is weak against $X$ errors on them. What if each of those three qubits were not a single physical qubit, but a **logical qubit of the bit-flip code**? An $X$ error would be caught inside its block. Encoding a code inside another code is called **concatenation**.
</details>

<details>
<summary>Hint 3: how many qubits, and what does a Z error do now?</summary>

Three blocks of three: nine physical qubits. A $Z$ error on any one qubit of a bit-flip block acts as a logical $Z$ on that whole block (Chapter 5, first cell), which means a phase flip of one qubit of the *outer* phase-flip code, which the outer code corrects. The outer checks compare the signs of blocks, so they look like $X^{\otimes 6}$ on two blocks.
</details>

<details>
<summary>Hint 4: how to build the encoder</summary>

Apply the phase-flip encoder to qubits 0, 3 and 6. Then apply the bit-flip encoder to each block: (0, 1, 2), (3, 4, 5) and (6, 7, 8). Order matters, just as in Exercise 5.
</details>

<details>
<summary>Hint 5: can you do it with fewer qubits?</summary>

Yes. Nine is not the minimum. Seven qubits are enough if you start from a better classical code than repetition (Chapters 9 and 10), and the smallest possible code uses five. A counting argument shows why five is the floor: one logical qubit needs $2$ dimensions, and each of the $1 + 3n$ possible "error or no error" cases needs its own copy of them, so $2\,(1 + 3n) \le 2^n$, which first holds at $n = 5$.
</details>

### How to test a code: the Knill-Laflamme conditions

How do we check *any* proposed code without simulating every error and decoder? In 1997 Knill and Laflamme gave an exact criterion. A code with logical states $|0_L\rangle, |1_L\rangle$ can correct a set of errors $\{E_a\}$ if and only if, for every pair of errors,

$$\langle i_L|\,E_a^\dagger E_b\,|j_L\rangle = c_{ab}\,\delta_{ij},$$

where $c_{ab}$ does not depend on $i, j$. In words: no pair of errors can turn $|0_L\rangle$ into something overlapping $|1_L\rangle$, and no pair of errors can *tell the difference* between $|0_L\rangle$ and $|1_L\rangle$. Otherwise the environment that caused the error would be learning the logical state.

The checker below applies the condition to all single-qubit Paulis on an encoder whose input is qubit 0. We test it on the two codes we have.

In [ ]:
def single_qubit_paulis(n):
    errs = {"I": Pauli("I" * n)}
    for q in range(n):
        for P in "XYZ":
            label = ["I"] * n
            label[n - 1 - q] = P
            errs[f"{P}{q}"] = Pauli("".join(label))
    return errs


def knill_laflamme(encoder):
    # True if the code (input on qubit 0) corrects every single-qubit error.
    n = encoder.num_qubits
    zero = Statevector(QuantumCircuit(n).compose(encoder))
    prep_one = QuantumCircuit(n); prep_one.x(0)
    one = Statevector(prep_one.compose(encoder))
    errs = single_qubit_paulis(n)
    A0 = np.array([zero.evolve(E).data for E in errs.values()])
    A1 = np.array([one.evolve(E).data for E in errs.values()])
    G00, G11, G01 = A0.conj() @ A0.T, A1.conj() @ A1.T, A0.conj() @ A1.T
    return np.allclose(G00, G11) and np.allclose(G01, 0)

for name, enc in [("bit-flip code", bitflip_encoder()), ("phase-flip code", phaseflip_encoder())]:
    print(f"{name}: corrects all single-qubit errors? {knill_laflamme(enc)}")

# Why the bit-flip code fails: Z_0 can tell |0_L> and |1_L> apart, so an environment causing Z errors learns the state.
zero = Statevector(bitflip_encoder())
prep1 = QuantumCircuit(3); prep1.x(0); one = Statevector(prep1.compose(bitflip_encoder()))
Z0 = Pauli("IIZ")
print(f"\nbit-flip code: <0_L|Z0|0_L> = {zero.expectation_value(Z0).real:+.0f},  <1_L|Z0|1_L> = {one.expectation_value(Z0).real:+.0f}  (must be equal)")

Both fail, as expected. The last line shows the bit-flip code's problem in one number: $Z_0$ has a different expectation value on the two logical states, which violates the condition with $E_a = I$, $E_b = Z_0$. The phase-flip code fails the same way with $X$ errors.

### Exercise 7: your own code

**Goal.** Build an encoder `ex7_encoder` on any number of qubits up to 9, with the state to protect entering on qubit 0 and all other qubits starting in $|0\rangle$, that passes the Knill-Laflamme check. Use the hints.

In [ ]:
ex7_encoder = None

In [ ]:
check("your code corrects every single-qubit error", lambda: knill_laflamme(ex7_encoder))

<details>
<summary>Solution: Shor's nine-qubit code</summary>

```python
ex7_encoder = QuantumCircuit(9, name="Shor")
# outer code: phase-flip code on qubits 0, 3, 6
ex7_encoder.cx(0, 3); ex7_encoder.cx(0, 6)
ex7_encoder.h([0, 3, 6])
# inner code: a bit-flip code inside each block
for b in (0, 3, 6):
    ex7_encoder.cx(b, b + 1); ex7_encoder.cx(b, b + 2)
```

The logical states are

$$|0_L\rangle = \tfrac{1}{2\sqrt2}\big(|000\rangle+|111\rangle\big)^{\otimes3}, \qquad |1_L\rangle = \tfrac{1}{2\sqrt2}\big(|000\rangle-|111\rangle\big)^{\otimes3}.$$

It uses **eight** checks: six $ZZ$ parities (two per block, catching $X$ errors) and two $X^{\otimes 6}$ parities comparing the signs of neighbouring blocks (catching $Z$ errors). This is the code Peter Shor published in 1995, the first quantum error-correcting code.

One curious feature: $Z_0$ and $Z_1$ have the same effect on the code (both flip the sign of block 0), so the code cannot tell them apart, yet it still corrects both, because applying either fix works. Codes in which different errors are indistinguishable but equally correctable are called **degenerate**, and they have no classical counterpart.
</details>

Further reading: P. Shor, [Scheme for reducing decoherence in quantum computer memory](https://doi.org/10.1103/PhysRevA.52.R2493), Phys. Rev. A 52 (1995); E. Knill and R. Laflamme, [Theory of quantum error-correcting codes](https://arxiv.org/abs/quant-ph/9604034) (1997).

## Chapter 8. The stabiliser language

Before the Steane code, a better vocabulary. Every code so far was defined by a set of **parity checks**, each a product of Pauli operators:

| code | checks |
|---|---|
| bit-flip | $Z_0Z_1,\ Z_1Z_2$ |
| phase-flip | $X_0X_1,\ X_1X_2$ |
| Shor | six $ZZ$ checks, two $X^{\otimes 6}$ checks |

These checks have three properties that make them powerful.

1. **They all commute with each other**, so they can be measured together without disturbing one another.
2. **Every codeword is a $+1$ eigenstate of every check.** The code space is *defined* as the set of states with all checks equal to $+1$. The checks are said to **stabilise** the code, and they are called **stabilisers**.
3. **The syndrome of a Pauli error is determined by commutation.** If an error $E$ anticommutes with a check $S$, then $S(E|\psi\rangle) = -ES|\psi\rangle = -E|\psi\rangle$, so the check reads $-1$. If they commute, it reads $+1$. Two Paulis anticommute exactly when they disagree (one $X$, the other $Z$ or $Y$, and so on) on an odd number of qubits.

So the whole error-correction analysis becomes bookkeeping with Pauli strings, which Qiskit's `Pauli` class does for us. Remember that Qiskit Pauli labels are written with **qubit 0 on the right**.

> **Physics note.** Daniel Gottesman developed this **stabiliser formalism** in his 1997 PhD thesis. Its other great consequence, the Gottesman-Knill theorem, says that circuits built only from H, S, CNOT and Pauli measurements can be simulated efficiently on a classical computer. That is why Aer's `stabilizer` method, which we use in Chapter 10, can simulate error-correction circuits on hundreds or thousands of qubits.

In [ ]:
bitflip_stabs = [Pauli("IZZ"), Pauli("ZZI")]          # Z0 Z1 and Z1 Z2 (qubit 0 on the right)

def syndrome(error, stabilisers):
    return tuple(int(error.anticommutes(S)) for S in stabilisers)

print("stabilisers commute with each other?", bitflip_stabs[0].commutes(bitflip_stabs[1]))

psi = Statevector(QuantumCircuit(3).compose(bitflip_encoder()))
print("<S> on the code state:", [round(psi.expectation_value(S).real, 3) for S in bitflip_stabs])
print()
for label in ["IIX", "IXI", "XII", "IIZ"]:
    print(f"error {label}:  syndrome (Z0Z1, Z1Z2) = {syndrome(Pauli(label), bitflip_stabs)}")

The Pauli bookkeeping reproduces the table from Chapter 3, and shows again that $Z$ errors are invisible to this code.

### Logical operators

A **logical operator** is a Pauli that commutes with every stabiliser (so it maps the code space to itself) but is not itself a stabiliser (so it acts non-trivially on the encoded qubit). For the bit-flip code, $\bar X = X_0X_1X_2$ and $\bar Z = Z_0$ (or $Z_1$, or $Z_2$; they differ by stabilisers). The weight of the smallest logical operator is the code's **distance** $d$. A code with distance $d$ can correct any $\lfloor (d-1)/2\rfloor$ errors. In the standard notation, a code on $n$ physical qubits encoding $k$ logical qubits with distance $d$ is written $[[n, k, d]]$.

| code | $[[n,k,d]]$ | why |
|---|---|---|
| bit-flip | $[[3,1,1]]$ | $\bar Z = Z_0$ has weight 1 |
| Shor | $[[9,1,3]]$ | smallest logical operators have weight 3 |
| Steane | $[[7,1,3]]$ | next chapter |

### Exercise 8: syndrome of a Y error

**Goal.** Without running the circuit, use `syndrome` and the bit-flip stabilisers to find the syndrome of a $Y$ error on qubit 1. Store the tuple in `ex8_syndrome`. Then explain in a comment why the bit-flip code still *fails* on this error even though the syndrome is not trivial.

<details>
<summary>Hint</summary>

The label for $Y$ on qubit 1 of three is `"IYI"`. After the decoder applies its correction, what is left of $Y_1$?
</details>

In [ ]:
ex8_syndrome = None

In [ ]:
check("syndrome of Y on qubit 1", lambda: tuple(ex8_syndrome) == (1, 1))

<details>
<summary>Solution</summary>

```python
ex8_syndrome = syndrome(Pauli("IYI"), bitflip_stabs)    # (1, 1)
# The decoder sees "X on qubit 1" and applies X_1. What remains is X_1 Y_1 = i Z_1,
# which is a logical Z: an undetected phase flip.
```
</details>

## Chapter 9. The Steane code: theory

### Start from a classical code

The Steane code is built from the classical **Hamming [7,4,3] code**: seven bits that carry four bits of data and correct any single bit flip. It is defined by three parity checks, the rows of the **parity-check matrix**

$$H = \begin{pmatrix}
0&0&0&1&1&1&1\\
0&1&1&0&0&1&1\\
1&0&1&0&1&0&1
\end{pmatrix}.$$

Look at the columns: column $j$ is the number $j$ written in binary ($1 = 001$, $2 = 010$, ..., $7 = 111$). A codeword is any 7-bit string $c$ with $Hc = 0 \pmod 2$. If bit $j$ flips, the syndrome $Hc'$ equals column $j$, which is **the position of the error written in binary**. Hamming's design could not be more elegant.

### Make it quantum: the CSS construction

Calderbank, Shor and Steane (1996) noticed that the Hamming code has a special property: its checks, the rows of $H$, are themselves Hamming codewords. That means we can use the same three rows **twice**:

- as **$Z$-type** stabilisers, a $Z$ on every qubit where the row has a 1. These catch $X$ errors, like the bit-flip code.
- as **$X$-type** stabilisers, an $X$ on every qubit where the row has a 1. These catch $Z$ errors, like the phase-flip code.

Number the qubits 0 to 6, so qubit $q$ is Hamming position $q+1$. The six stabilisers are

| row | support (qubits) | $Z$-type | $X$-type |
|---|---|---|---|
| $r_0$ | 0, 2, 4, 6 | $Z_0Z_2Z_4Z_6$ | $X_0X_2X_4X_6$ |
| $r_1$ | 1, 2, 5, 6 | $Z_1Z_2Z_5Z_6$ | $X_1X_2X_5X_6$ |
| $r_2$ | 3, 4, 5, 6 | $Z_3Z_4Z_5Z_6$ | $X_3X_4X_5X_6$ |

(Rows listed bottom-to-top of $H$, so that row $r_k$ gives bit $k$ of the binary position.)

They all commute: two $Z$-types commute trivially, as do two $X$-types, and any $Z$-type and $X$-type row overlap on an **even** number of qubits (0, 2 or 4), because the rows of $H$ are orthogonal mod 2. That evenness is exactly the CSS condition.

Six independent stabilisers on seven qubits leave $7 - 6 = 1$ logical qubit. The logical operators are

$$\bar X = X^{\otimes 7}, \qquad \bar Z = Z^{\otimes 7},$$

and both can be reduced to weight 3 by multiplying with stabilisers (for example $X_2X_4X_5$). No weight-1 or weight-2 Pauli commutes with all six checks without being trivial, so the distance is 3: **the Steane code is a $[[7,1,3]]$ code and corrects any single-qubit error.**

### Decoding is Hamming decoding, twice

- An $X$ error on qubit $q$ anticommutes with the $Z$-type checks whose rows contain $q$. The three bits spell out $q+1$ in binary.
- A $Z$ error on qubit $q$ does the same with the $X$-type checks.
- A $Y$ error trips both, pointing to the same qubit, and gets both corrections.

The $X$ and $Z$ parts are corrected **independently**, which is what Hint 1 in Chapter 7 asked for.

> **Physics note.** Andrew Steane published this code in 1996, and Calderbank and Shor generalised it the same year into the family now called **CSS codes**: take any classical code that contains its own dual, and you get a quantum code. The surface code that Google, IBM and others run on hardware today is also a CSS code, with $Z$-type and $X$-type checks on a two-dimensional grid. The Steane code's special charm is that many logical gates are **transversal**: applying a gate to all seven qubits in parallel applies it to the logical qubit, so an error on one qubit can never spread to another within the same block. We test this below.

Let us check every claim above with Pauli bookkeeping.

In [ ]:
steane_rows = [[0, 2, 4, 6], [1, 2, 5, 6], [3, 4, 5, 6]]

def pauli_on(P, qubits, n=7):
    label = ["I"] * n
    for q in qubits:
        label[n - 1 - q] = P
    return Pauli("".join(label))

Z_checks = [pauli_on("Z", r) for r in steane_rows]
X_checks = [pauli_on("X", r) for r in steane_rows]
stabilisers = Z_checks + X_checks
X_L, Z_L = Pauli("X" * 7), Pauli("Z" * 7)

print("all stabilisers commute:      ", all(a.commutes(b) for a in stabilisers for b in stabilisers))
print("logical X, Z commute with all:", all(L.commutes(S) for L in (X_L, Z_L) for S in stabilisers))
print("logical X and Z anticommute:  ", X_L.anticommutes(Z_L))
print()
print(" error | Z-check syndrome -> position | X-check syndrome -> position")
for q in [0, 3, 6]:
    for P in "XZY":
        E = pauli_on(P, [q])
        sx = sum(int(E.anticommutes(S)) << k for k, S in enumerate(Z_checks))
        sz = sum(int(E.anticommutes(S)) << k for k, S in enumerate(X_checks))
        print(f"  {P}{q}   |   {sx:03b} -> {sx}                     |   {sz:03b} -> {sz}")

Every $X$ (and $Y$) error on qubit $q$ produces the $Z$-check syndrome $q+1$ in binary, and every $Z$ (and $Y$) error produces the same number from the $X$-checks. Hamming's trick, twice over.

### Exercise 9: a smaller logical operator

**Goal.** $\bar X = X^{\otimes 7}$ has weight 7. Find a **weight-3** logical $X$, different from $X_2X_4X_5$, and store the list of its qubits in `ex9_support`.

<details>
<summary>Hint</summary>

A Pauli $X$-string is logical if it commutes with every $Z$-check (its support is a Hamming codeword: the binary positions XOR to zero) and is not a product of $X$-checks (it has odd weight). Positions 1, 2, 3 are binary 001, 010, 011. What do they XOR to?
</details>

In [ ]:
ex9_support = None      # e.g. [a, b, c] with qubit indices 0..6

In [ ]:
def _ex9():
    L = pauli_on("X", ex9_support)
    return (len(set(ex9_support)) == 3 and sorted(ex9_support) != [2, 4, 5]
            and all(L.commutes(S) for S in stabilisers) and L.anticommutes(Z_L))

check("a new weight-3 logical X", _ex9)

<details>
<summary>Solution</summary>

```python
ex9_support = [0, 1, 2]       # Hamming positions 1, 2, 3: 001 ^ 010 ^ 011 = 000
```

There are seven weight-3 Hamming codewords (they are the lines of the Fano plane), and each gives a weight-3 logical $X$. Their existence is why the distance is 3 and not higher: two $X$ errors on qubits 0 and 1 produce the syndrome $1 \oplus 2 = 3$, the decoder "fixes" qubit 2, and the result is $X_0X_1X_2$, a logical $X$. You will see that failure happen in Chapter 10.
</details>

Further reading: A. Steane, [Multiple particle interference and quantum error correction](https://arxiv.org/abs/quant-ph/9601029), Proc. R. Soc. A 452 (1996); A. R. Calderbank and P. Shor, [Good quantum error-correcting codes exist](https://arxiv.org/abs/quant-ph/9512032) (1996).

## Chapter 10. The Steane code in Qiskit

We now build the complete circuit: encoder, error, syndrome extraction with six ancillas, decoder and correction, and verification.

### The encoder

$|0_L\rangle$ is the equal superposition of the eight strings in the row space of $H$ (the code generated by the three rows). Preparing it is the GHZ trick from Essentials Chapter 6, three times over: for each row, pick a **pivot** qubit that appears in that row and in no other row, put the pivot in $|+\rangle$, and CNOT it onto the rest of its row. The qubits 0, 1 and 3 (Hamming positions 1, 2, 4, the powers of two) are pivots for rows $r_0, r_1, r_2$.

To encode an arbitrary state, put it on qubit 2 and first spread it with CNOTs along the weight-3 logical operator $X_2X_4X_5$. Then $|0\rangle$ stays $|0\cdots0\rangle$ and $|1\rangle$ becomes the logical-$X$ string, and the pivot step turns both into the correct code states.

In [ ]:
PIVOTS = [0, 1, 3]
INPUT = 2

def steane_encoder():
    qc = QuantumCircuit(7, name="Steane enc")
    qc.cx(INPUT, 4); qc.cx(INPUT, 5)            # |1> -> X_2 X_4 X_5 |0...0>
    qc.barrier()
    qc.h(PIVOTS)
    for p, row in zip(PIVOTS, steane_rows):
        for q in row:
            if q != p:
                qc.cx(p, q)
    return qc

steane_encoder().draw("mpl", fold=-1)

Before building on it, we test it. Every stabiliser should have expectation $+1$ on any encoded state, and the logical operators should report the state we put in.

In [ ]:
def encode(prep_gates):
    qc = QuantumCircuit(7)
    for g in prep_gates:
        getattr(qc, g)(INPUT)
    return Statevector(qc.compose(steane_encoder()))

print(" input | six stabilisers           | <Z_L>  <X_L>")
for name, prep in [("|0>", []), ("|1>", ["x"]), ("|+>", ["h"]), ("|->", ["x", "h"])]:
    sv = encode(prep)
    st = [round(sv.expectation_value(S).real) for S in stabilisers]
    print(f" {name:5s} | {st} | {sv.expectation_value(Z_L).real:+.2f}  {sv.expectation_value(X_L).real:+.2f}")

zero_L = encode([])
print("\n|0_L> is a superposition of", len(zero_L.probabilities_dict()), "bitstrings:")
print(sorted(str(k) for k in zero_L.probabilities_dict()))

All six stabilisers read $+1$ for every input, $\bar Z$ distinguishes $|0\rangle$ from $|1\rangle$, and $\bar X$ distinguishes $|+\rangle$ from $|-\rangle$. The eight bitstrings of $|0_L\rangle$ are the row space of $H$: the zero string and seven strings of weight 4.

### Syndrome extraction

Each $Z$-type check is measured exactly as in the bit-flip code (CNOTs from data into an ancilla in $|0\rangle$). Each $X$-type check is measured as in the phase-flip code (ancilla in $|+\rangle$, CNOTs from the ancilla into the data, Hadamard, measure). Six ancillas in total.

In [ ]:
def steane_syndrome_extraction(qc, data, anc_z, anc_x, syn_x, syn_z):
    # Z-type checks: detect X errors. Result in syn_x.
    for k, row in enumerate(steane_rows):
        for q in row:
            qc.cx(data[q], anc_z[k])
    qc.measure(anc_z, syn_x)
    qc.barrier()
    # X-type checks: detect Z errors. Result in syn_z.
    qc.h(anc_x)
    for k, row in enumerate(steane_rows):
        for q in row:
            qc.cx(anc_x[k], data[q])
    qc.h(anc_x)
    qc.measure(anc_x, syn_z)
    qc.barrier()

data, anc_z, anc_x = QuantumRegister(7, "data"), QuantumRegister(3, "anc_z"), QuantumRegister(3, "anc_x")
syn_x, syn_z = ClassicalRegister(3, "syn_x"), ClassicalRegister(3, "syn_z")
demo = QuantumCircuit(data, anc_z, anc_x, syn_x, syn_z)
steane_syndrome_extraction(demo, data, anc_z, anc_x, syn_x, syn_z)
demo.draw("mpl", fold=-1, scale=0.6)

### The decoder

Because the syndrome *is* the binary position, the decoder is one line per qubit: if the 3-bit register reads $s \neq 0$, apply the correction to qubit $s - 1$. We do this separately for `syn_x` (apply $X$) and `syn_z` (apply $Z$).

### The full cycle

Now the whole thing, built as one function. The errors are given in the same format as in Chapter 4. As before, `id` gates on the data mark where random noise will act, and at the end we decode and check that all seven data qubits come back to $|0\rangle$.

In [ ]:
def steane_cycle(prep_gates=(), errors=(), correct=True, blocks=False):
    # blocks=True draws the encoder and syndrome extraction as labelled boxes (for display only)
    data  = QuantumRegister(7, "data")
    anc_z = QuantumRegister(3, "anc_z")
    anc_x = QuantumRegister(3, "anc_x")
    syn_x = ClassicalRegister(3, "syn_x")
    syn_z = ClassicalRegister(3, "syn_z")
    out   = ClassicalRegister(7, "out")
    qc = QuantumCircuit(data, anc_z, anc_x, syn_x, syn_z, out)

    # 1. prepare the logical state and encode
    for name, *params in prep_gates:
        getattr(qc, name)(*params, data[INPUT])
    if blocks:
        qc.append(steane_encoder().to_instruction(label="encode"), data)
    else:
        qc.compose(steane_encoder(), data, inplace=True)
    qc.barrier()

    # 2. errors
    for name, q, *params in errors:
        getattr(qc, name)(*params, data[q])
    qc.id(data)
    qc.barrier()

    # 3. syndrome extraction
    if blocks:
        sub = QuantumCircuit(data, anc_z, anc_x, syn_x, syn_z)
        steane_syndrome_extraction(sub, data, anc_z, anc_x, syn_x, syn_z)
        qc.append(sub.to_instruction(label="syndrome"), [*data, *anc_z, *anc_x], [*syn_x, *syn_z])
    else:
        steane_syndrome_extraction(qc, data, anc_z, anc_x, syn_x, syn_z)

    # 4. decoding and correction: syndrome value s means "qubit s - 1"
    if correct:
        for s in range(1, 8):
            with qc.if_test((syn_x, s)):
                qc.x(data[s - 1])
            with qc.if_test((syn_z, s)):
                qc.z(data[s - 1])
    qc.barrier()

    # 5. verification: unencode, undo the preparation, everything should read 0
    if blocks:
        qc.append(steane_encoder().to_instruction(label="decode").inverse(), data)
    else:
        qc.compose(steane_encoder().inverse(), data, inplace=True)
    for name, *params in reversed(prep_gates):
        inv = QuantumCircuit(1); getattr(inv, name)(*params, 0)
        qc.compose(inv.inverse(), [data[INPUT]], inplace=True)
    qc.measure(data, out)
    return qc

full = steane_cycle(prep_gates=[("ry", 1.1), ("rz", 0.7)], errors=[("y", 4)])
print("qubits:", full.num_qubits, "  depth:", full.depth(), "  operations:", dict(full.count_ops()))

print("Overview, with the encoder and syndrome extraction drawn as boxes:")
display(steane_cycle(prep_gates=[("ry", 1.1), ("rz", 0.7)], errors=[("y", 4)], blocks=True).draw("mpl", fold=-1, scale=0.6))

And the same circuit gate by gate. Between the barriers, from left to right: preparation and encoder, the error slot, the $Z$-type checks and their measurement, the $X$-type checks and their measurement, the 14 conditional corrections, and the verification.

In [ ]:
full.draw("mpl", fold=-1, scale=0.45)

That is how the Steane code looks in Qiskit: 7 data qubits, 6 ancillas, 24 CNOTs for syndrome extraction, and 14 classically controlled corrections.

> **Predict.** We will now inject every single-qubit Pauli error, $X$, $Y$ or $Z$ on each of the seven qubits (21 cases plus "no error"), on a random-looking logical state. What fraction of the cases should be corrected?

In [ ]:
prep = [("ry", 1.1), ("rz", 0.7)]
cases = [[]] + [[(P, q)] for q in range(7) for P in ("x", "y", "z")]
circuits = [steane_cycle(prep, err) for err in cases]
results = aer_sampler.run(circuits, shots=200).result()

print(" error | syn_x | syn_z | P(recovered)")
print("-------+-------+-------+-------------")
n_ok = 0
for err, res in zip(cases, results):
    sx = list(res.data.syn_x.get_counts())[0]
    sz = list(res.data.syn_z.get_counts())[0]
    p_ok = res.data.out.get_counts().get("0" * 7, 0) / 200
    n_ok += p_ok == 1
    label = "none" if not err else f"{err[0][0].upper()}{err[0][1]}"
    print(f" {label:5s} |  {sx}  |  {sz}  | {p_ok:.2f}")
print(f"\nCorrected {n_ok} of {len(cases)} cases.")

All 22 cases are corrected, and the syndrome columns spell out the qubit positions in binary. Now the failure mode from Exercise 9: two $X$ errors on qubits 0 and 1.

In [ ]:
res = aer_sampler.run([steane_cycle([], [("x", 0), ("x", 1)])], shots=200).result()[0].data   # start from |0_L>
print("syn_x:", res.syn_x.get_counts(), "  -> decoder blames qubit", int(list(res.syn_x.get_counts())[0], 2) - 1)
print("verification outcomes:", res.out.get_counts())

The decoder blames qubit 2, the result is the logical operator $X_0X_1X_2$, and the verification finds qubit 2 (the input position) in $|1\rangle$: the logical $|0_L\rangle$ has silently become $|1_L\rangle$. Distance 3 means one error is always fine and two can be fatal.

### Transversal gates

> **Predict.** Apply $H$ to all seven qubits of $|0_L\rangle$. Which logical state do you get? And what does $S$ on all seven qubits do?

In [ ]:
zero_L, one_L, plus_L = encode([]), encode(["x"]), encode(["h"])

H7 = QuantumCircuit(7); H7.h(range(7))
S7 = QuantumCircuit(7); S7.s(range(7))

print("|<+_L| H^7 |0_L>|   =", round(abs(plus_L.inner(zero_L.evolve(H7))), 4))
print("<0_L| S^7 |0_L>     =", np.round(zero_L.inner(zero_L.evolve(S7)), 4))
print("<1_L| S^7 |1_L>     =", np.round(one_L.inner(one_L.evolve(S7)), 4))

$H^{\otimes 7}$ is a logical Hadamard: it swaps the $Z$-type and $X$-type stabilisers, which are identical by construction. That symmetry is special to codes like Steane's where the same classical code is used twice.

### Exercise 10: what does $S^{\otimes 7}$ do?

**Goal.** From the output above, decide which logical gate $S^{\otimes 7}$ implements. Store `"S"` or `"Sdg"` in `ex10_gate`.

<details>
<summary>Hint</summary>

$S = \mathrm{diag}(1, i)$ and $S^\dagger = \mathrm{diag}(1, -i)$. The cell printed the phases picked up by $|0_L\rangle$ and $|1_L\rangle$.
</details>

In [ ]:
ex10_gate = None

In [ ]:
check("the logical gate implemented by S on every qubit", lambda: ex10_gate == "Sdg")

<details>
<summary>Solution</summary>

```python
ex10_gate = "Sdg"
```

$|1_L\rangle$ is a superposition of weight-3 and weight-7 strings (the odd Hamming codewords), and $i^3 = i^7 = -i$. So $S^{\otimes 7}$ gives $|1_L\rangle \to -i|1_L\rangle$, which is logical $S^\dagger$. To get logical $S$, apply $S^\dagger$ to every qubit. Together with transversal $H$ and transversal CNOT between two Steane blocks, this gives the whole Clifford group transversally. The one gate missing for universality is $T$, which needs a separate technique called **magic-state distillation**.
</details>

### How good is the logical qubit?

Finally, random noise. Every data qubit independently suffers $X$, $Y$ or $Z$, each with probability $p/3$ (a depolarising error of strength $p$). We measure the logical failure rate on $|0_L\rangle$ (sensitive to logical $X$) and on $|+_L\rangle$ (sensitive to logical $Z$), and compare with an unprotected qubit, which fails with probability $2p/3$ in either case.

Because the circuit contains only Clifford gates and Pauli noise, Aer's **stabilizer** simulator handles it very quickly, so we can afford many shots.

An important caveat: here the noise acts only *between* encoding and syndrome measurement. The syndrome extraction itself is assumed perfect. Making it robust against its own faults is the subject of **fault tolerance**, introduced below.

In [ ]:
ps = np.array([0.01, 0.02, 0.04, 0.06, 0.1, 0.15, 0.2, 0.3])
shots = 20000
fail_0, fail_plus = [], []
for p in ps:
    smp = noisy_sampler(p_x=p / 3, p_y=p / 3, p_z=p / 3, method="stabilizer")
    for prep, store in [([], fail_0), ([("h",)], fail_plus)]:
        out = smp.run([steane_cycle(prep)], shots=shots).result()[0].data.out.get_counts()
        store.append(1 - out.get("0" * 7, 0) / shots)

fine = np.linspace(0.005, 0.3, 200)
bound = 1 - (1 - fine)**7 - 7 * fine * (1 - fine)**6        # P(two or more qubits hit)

plt.figure(figsize=(7.5, 4.2))
plt.loglog(fine, 2 * fine / 3, "--", c="gray", label="unprotected qubit: $2p/3$")
plt.loglog(fine, bound, ":", c="k", label="P(2 or more errors), an upper bound")
plt.loglog(ps, fail_0, "o-", label=r"Steane, $|0_L\rangle$ (logical X failures)")
plt.loglog(ps, fail_plus, "s-", label=r"Steane, $|+_L\rangle$ (logical Z failures)")
plt.xlabel("physical error probability p"); plt.ylabel("logical failure probability")
plt.title("The Steane code under depolarising noise"); plt.legend(fontsize=8); plt.grid(alpha=.3, which="both")
plt.show()

for p, a, b in zip(ps, fail_0, fail_plus):
    print(f"p = {p:.2f}:  unprotected {2*p/3:.4f}   Steane |0_L> {a:.4f}   |+_L> {b:.4f}")

On the log-log plot the Steane curves have **slope 2**, because failure needs two errors, while the unprotected qubit has slope 1. The curves cross at the **pseudo-threshold**: below it, encoding helps, and the further below, the more it helps. Some two-error combinations happen to be harmless (an $X$ on one qubit and a $Z$ on another, for instance, are corrected independently), which is why the simulated curves lie below the "two or more errors" bound.

> **Physics note.** The **threshold theorem** (Aharonov and Ben-Or, Kitaev, Knill, Laflamme and Zurek, around 1997) says that if every component, including the syndrome extraction itself, fails with probability below some threshold, then concatenating codes, or growing a code's distance, suppresses the logical error rate as far as you like with only polylogarithmic overhead. Making the extraction fault tolerant needs care: in our circuit a single fault on one ancilla can spread through its four CNOTs to several data qubits. Remedies include Shor-style cat-state ancillas, Steane-style extraction with encoded ancillas, and **flag qubits**. In December 2024 Google reported a surface-code logical qubit whose error rate fell by roughly a factor of two each time the code distance increased, the first clear experimental demonstration of operating below threshold. IBM is pursuing **qLDPC codes** such as the $[[144, 12, 12]]$ "gross" code, which encode many more logical qubits per physical qubit than the surface code.

Further reading: the [Foundations of quantum error correction](https://quantum.cloud.ibm.com/learning/en/courses/foundations-of-quantum-error-correction) course on IBM Quantum Learning, which covers CSS codes, the Steane code and fault tolerance in detail.

## Where we go next

| Topic | What it builds on |
|---|---|
| The five-qubit code $[[5,1,3]]$ | The smallest code correcting any single error (Hint 5); not CSS, so $X$ and $Z$ are not corrected separately |
| Fault-tolerant syndrome extraction | Chapter 10: flag qubits, cat-state ancillas, Steane-style extraction, repeated syndrome rounds |
| The surface code | A CSS code with weight-4 checks on a 2D grid of nearest neighbours; today's leading hardware code |
| qLDPC codes | Codes with few checks per qubit but many logical qubits, such as IBM's bivariate bicycle codes |
| Decoders | Lookup tables (this notebook), minimum-weight perfect matching, union-find, belief propagation with OSD; real-time decoders are often implemented on FPGAs or ASICs |
| Universal fault-tolerant computation | Transversal Cliffords (Exercise 10) plus magic-state distillation for the $T$ gate |
| Connection to the algorithms notebooks | Syndrome measurement is phase kickback (Deutsch-Jozsa, Chapter 3) and one-bit phase estimation (QPE) of a stabiliser with eigenvalues $\pm1$ |

## Quick reference

```python
# Codes in this notebook (input qubit -> code space)
bitflip_encoder()       # |0> -> |000>, |1> -> |111>                   checks Z0Z1, Z1Z2
phaseflip_encoder()     # |0> -> |+++>, |1> -> |--->                   checks X0X1, X1X2
steane_encoder()        # input on qubit 2, 7 qubits                    checks: rows of Hamming H, as Z and as X

# Measuring checks with an ancilla
qc.cx(data_i, anc); qc.cx(data_j, anc)                    # Z_i Z_j  (ancilla starts in |0>)
qc.h(anc); qc.cx(anc, data_i); qc.cx(anc, data_j); qc.h(anc)   # X_i X_j
qc.measure(anc, syn)

# Classically controlled correction (dynamic circuits)
with qc.if_test((syn, value)):
    qc.x(data[q])

# Pauli bookkeeping
Pauli("IZZ").anticommutes(Pauli("IXI"))    # does this error trip this check?  (qubit 0 on the right)
Statevector(...).expectation_value(S)     # +1 on code states

# Simulation
AerSampler()                                              # mid-circuit measurement and if_test
NoiseModel().add_all_qubit_quantum_error(pauli_error([...]), ["id"])   # noise only where you put id gates
AerSampler(options=dict(backend_options=dict(method="stabilizer")))     # fast for Clifford circuits
```

## Further reading

IBM Quantum:
- [Foundations of quantum error correction](https://quantum.cloud.ibm.com/learning/en/courses/foundations-of-quantum-error-correction) (IBM Quantum Learning)
- [Classical feedforward and control flow](https://quantum.cloud.ibm.com/docs/en/guides/classical-feedforward-and-control-flow), for `if_test` and dynamic circuits
- [Build noise models](https://quantum.cloud.ibm.com/docs/en/guides/build-noise-models)

Original papers:
- P. Shor, [Scheme for reducing decoherence in quantum computer memory](https://doi.org/10.1103/PhysRevA.52.R2493) (1995)
- A. Steane, [Multiple particle interference and quantum error correction](https://arxiv.org/abs/quant-ph/9601029) (1996)
- A. R. Calderbank and P. Shor, [Good quantum error-correcting codes exist](https://arxiv.org/abs/quant-ph/9512032) (1996)
- E. Knill and R. Laflamme, [Theory of quantum error-correcting codes](https://arxiv.org/abs/quant-ph/9604034) (1997)
- D. Gottesman, [Stabilizer codes and quantum error correction](https://arxiv.org/abs/quant-ph/9705052), PhD thesis (1997)

Textbooks: Nielsen and Chuang, *Quantum Computation and Quantum Information*, chapter 10; and D. Gottesman's lecture notes, [An introduction to quantum error correction and fault-tolerant quantum computation](https://arxiv.org/abs/0904.2557) (2009).